In [ ]:
import os
from ase import Atoms
from ase.units import Ry
from ase.calculators.siesta import Siesta
from ase.calculators.siesta.parameters import Species, PAOBasisBlock
from src.structure import Perovskite

In [ ]:
def generate_fdf(perovskite, xcf='PBEsol', basis='DZP', EnergyShift=0.01, SplitNorm=0.15,
                 MeshCutoff=200, kgrid=(10, 10, 10), dir=''):
    """Function to generate a Siesta input file (.fdf) for a given perovskite structure and calculation parameters.
    Parameters:
    - perovskite: Perovskite object containing the structure and properties of the material
    - xcf: Exchange-correlation functional to use (default: 'PBEsol')
    - basis: Basis set to use for the calculation (default: 'DZP')
    - EnergyShift: Energy shift for the basis functions in Ry (default: 0.01)
    - SplitNorm: Split norm for the basis functions (default: 0.15)
    - MeshCutoff: Mesh cutoff for the real-space grid in Ry (default: 200)
    - kgrid: Tuple specifying the k-point grid for the calculation (default: (10, 10, 10))
    - dir: Directory to save the input file (default: current working directory)
    """
    # Define current working directory and extract information from the perovskite object
    cwd = os.getcwd()
    formula = perovskite.formula
    atoms = perovskite.atoms
    symbols = atoms.symbols
    bulk = perovskite.bulk
    # Convert kgrid to a list to allow for modification
    kgrid = list(kgrid)

    species=[
        Species(symbol=symbols[0], basis_set=basis),
        Species(symbol=symbols[1], basis_set=basis),
        Species(symbol=symbols[2], basis_set=basis),
    ]

    # Calculation parameters in a dictionary
    calc_params = {
        'label': f'{formula}',
        'xc': xcf,
        #'basis_set': basis,
        'mesh_cutoff': MeshCutoff * Ry,
        'energy_shift': EnergyShift * Ry,
        'kpts': kgrid,
        'directory': dir,
        'pseudo_path': os.path.join(cwd, 'pseudos', f'{xcf}')
    }
    # fdf arguments in a dictionary
    fdf_args = {
        #'PAO.BasisSize': basis,
        'PAO.SplitNorm': SplitNorm,
        'SCF.DM.Tolerance': 1e-6,
    }
    if not bulk:
        # Add dipole correction for slab calculations to avoid spurious interactions between periodic images
        fdf_args['Slab.DipoleCorrection'] = 'T'
    # Set up the Siesta calculator
    calc = Siesta(species=species, **calc_params, fdf_arguments=fdf_args)
    calc.write_input(atoms, 'energies')
    

In [ ]:
perov = Perovskite('BaTiO3', bulk=True)

In [ ]:
generate_fdf(perov, dir='resultsold/fdf')